## Deep Q-Learning on Lunar Lander

Same **API** as the tutorial notebook: `DeepQNetwork`, `ReplayBuffer`, `DQNAgent.act` / `step` / `update_model`. Code lives in **`dqn/src/DQN/`**.

**Set the Jupyter working directory to the repository root** (the folder that contains `dqn/`).

Uses **Gymnasium** `LunarLander-v3` (Box2D).


### Setup


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
SRC = REPO_ROOT / "dqn" / "src"
assert SRC.is_dir(), f"Expected {SRC} — cwd should be repo root"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import gymnasium as gym

from DQN.agent import DQNAgent
from DQN.config import DQNConfig
from DQN.train import train
from DQN.utils import plot_training_curves, resolve_device, save_results


### Random policy sanity check


In [ ]:
env = gym.make("LunarLander-v3")
env.reset(seed=42)
while True:
    _, _, terminated, truncated, _ = env.step(env.action_space.sample())
    if terminated or truncated:
        break
env.close()
print("OK")


### Train DQN


In [ ]:
env = gym.make("LunarLander-v3")
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

config = DQNConfig(
    num_episodes=200,
    max_steps=500,
    batch_size=64,
    buffer_size=10_000,
    target_update_episodes=10,
    hidden_size=64,
)
print("Device:", resolve_device(config.device))

agent = DQNAgent(state_dim, action_dim, config)
scores, losses = train(env, agent, config)

save_dir = REPO_ROOT / "results" / "dqn"
save_results(scores, losses, save_dir=str(save_dir))
plot_training_curves(scores, losses, save_dir=str(save_dir))
agent.save(str(save_dir / "model_notebook.pth"))
env.close()
print("Saved to", save_dir)


### Greedy rollout (epsilon = 0)


In [ ]:
env = gym.make("LunarLander-v3")
agent = DQNAgent(state_dim, action_dim, config)
agent.load(str(save_dir / "model_notebook.pth"))
total = 0.0
state, _ = env.reset(seed=0)
while True:
    a = agent.act(state, 0.0)
    state, r, term, trunc, _ = env.step(a)
    total += float(r)
    if term or trunc:
        break
env.close()
print("Greedy episode return:", total)
